In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

# 1. Load the Models
# Stage 1: Face Detector (YOLO11n-face)
# Note: 'yolo11n-face.pt' on HuggingFace (e.g., AdamCodd/YOLO11n-face-detection)
face_model = YOLO(r'/Users/kethniimasha/Documents/Yolo training/yolo11n.pt') 

# Stage 2: Emotion Classifier (Your custom model)
emotion_model = YOLO(r"/Users/kethniimasha/Desktop/FYP Final Project/runs/classify/YOLO best results/weights/best.pt")

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # --- STAGE 1: FACE DETECTION ---
    # Run face detection on the full frame
    # convert to grayscale for better performance (optional, depends on your face model)
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray_3ch = cv2.cvtColor(gray_frame, cv2.COLOR_GRAY2BGR)
    face_results = face_model.predict(source=gray_3ch, stream=True, conf=0.5)
    annotated_frame = frame.copy()

    for r in face_results:
        for box in r.boxes:
            # Get coordinates: [x1, y1, x2, y2]
            b = box.xyxy[0].cpu().numpy().astype(int)
            x1, y1, x2, y2 = b

            # Extract the face crop
            face_crop = frame[y1:y2, x1:x2]

            if face_crop.size > 0:
                # --- STAGE 2: EMOTION CLASSIFICATION ---
                # Run your emotion model ONLY on the cropped face
                emotion_results = emotion_model.predict(source=face_crop, verbose=False)

                for er in emotion_results:
                    if er.probs:
                        class_id = er.probs.top1
                        label = er.names[class_id]
                        confidence = er.probs.top1conf.item()

                        # Visualizing the results
                        color = (0, 255, 0)
                        text = f"{label} ({confidence:.2f})"
                        
                        # Draw bounding box for the face
                        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)
                        
                        # Add label above the box
                        cv2.putText(annotated_frame, text, (x1, y1 - 10), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    cv2.imshow('YOLO11 Face + Emotion Detection', annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


0: 384x640 1 person, 54.3ms
Speed: 2.5ms preprocess, 54.3ms inference, 3.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 88.2ms
Speed: 2.0ms preprocess, 88.2ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 158.4ms
Speed: 2.0ms preprocess, 158.4ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 43.4ms
Speed: 1.9ms preprocess, 43.4ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 39.1ms
Speed: 1.4ms preprocess, 39.1ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 39.2ms
Speed: 1.4ms preprocess, 39.2ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 36.1ms
Speed: 1.9ms preprocess, 36.1ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 40.5ms
Speed: 1.5ms preprocess, 40.5ms inference, 0.5ms postprocess per image at shape (1, 3,